In [27]:
import json
import os
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool
from dotenv import load_dotenv, find_dotenv
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchResults


In [3]:
from langchain_openai import ChatOpenAI

load_dotenv( find_dotenv(), override=True )

# Verificar qual chave está sendo usada (mostra só os últimos 4 caracteres)
api_key = os.environ.get("OPENAI_API_KEY", "NÃO ENCONTRADA")
print(f"Chave carregada termina em: ...{api_key[-4:]}")


llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

Chave carregada termina em: ...js8A


In [10]:
csv_imoveis = CSVSearchTool(csv="files/imoveis.csv")

In [11]:
# Agente Corretor de Imóveis
corretor_imoveis = Agent(
    role="Corretor Imóveis",
    goal="Obtenha as preferências do cliente e busque imóveis compatíveis no banco de dados",
    backstory="Especialista no mercado imbiliário, encontra as melhores opções baseadas no perfil do cliente",
    verbose=True,
    max_iter=5,
    tools=[csv_imoveis],
    memory=True
)

In [12]:
# Tarefa -  Buscar Imóveis
buscar_imoveis = Task(
    description="Pesquise imóveis na região deseja pelo cliente considerando faixa de preço e tipo de imóvel",
    expected_output="Lista de imóveis disponíveis com detalhes sobre localização, preço e características",
    agent=corretor_imoveis
)

In [23]:
from crewai.tools import BaseTool

def obter_precos_imoveis(cidade: str = "geral"):
    precos = {
        "São Paulo": {"tendencia": "aumento", "percentual": 5.2},
        "Rio de Janeiro": {"tendencia": "estavel", "percentual": 0.0},
        "Belo Horizonte": {"tendencia": "queda", "percentual": -3.1},
        "geral": {"tendencia": "aumento", "percentual": 4.0},
    }
    return precos.get(cidade, precos["geral"])

class TendenciaPrecosImoveisTool(BaseTool):
    name: str = "Analisador de Preços Imbiliários"
    description: str = "Obtém tendências de preços de imóveis com base na cidade especificaada"

    def _run(self, cidade: str) -> dict:
        """
        Execita a análise re preço imobiliários e retorna a tendência com base na cidade
        """
        try:
            return obter_precos_imoveis(cidade)
        except Exception as e:
            return {"erro": f"Erro ao obter tendências de preços {str(e)}"}

In [14]:
analista_mercado = Agent(
    role="Analista de Mercado Imobiliário",
    goal="Analista tendências de preços e ajuda a prever a valorização ou desvalorização dos imóveis da cidade {cidade}",
    backstory="Experiente no setor, usa dados históricos para prever preços futuros.",
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [16]:
obter_tendencias = Task(
    description="""
    Analise o histórico de preços de imóveis na cidade {cidade} e forneça insights sobre
    valorização ou desvalorização. Considere o tipo de imóvel {tipo_imovel} e a 
    faixa de preço {faixa_preco}
    """,
    expected_output="Resumo da tendência dos preços no mercado imobiliário",
    tools=[TendenciaPrecosImoveisTool()],
    agent=analista_mercado,
    parameters=["cidade"]
)

In [18]:
analista_noticias = Agent(
    role="Analista de Notícias Imobiliários",
    goal="Busca notícias relevantes sobre o mercado imbobiliário para avaliar fatores externos.",
    backstory="Especialista em analisar notícias e tendências econômicas que afetam os preços dos imóveis",
    verbose=True,
    max_iter=5,
    memory=True
)

In [25]:
searchTool = DuckDuckGoSearchResults(backend="news", num_results=5)

In [26]:
searchTool

DuckDuckGoSearchResults(max_results=5, api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'), backend='news')

In [ ]:
buscar_noticias = Task(
    description=f"Pesquise notícias recentes sobre o mercado imobiliário. Data atual: {datetime.now}"
)